### Pasos para ejecutar este archivo

### instala python y demas dependencias 
- instalar python
- instalar python-venv
- instalar python-pip

### crear la carpeta del proyecto y abrirlo en vscode

### activar el entorno virtual
en Linux

 ```bash
python3 -m venv .venv
source .venv/bin/active
```

- crea el entorno virtual
- activalo

deberia aparecer algo como 

```bash
(.venv) usuario@pc:~/Documentos/AprendizajeComp$

```

### instala las dependencias para el notebook

```bash
# Actualizar pip dentro del entorno
python -m pip install --upgrade pip
# Instalar ipykernel (para que el venv pueda ser un kernel de Jupyter)
python -m pip install ipykernel

# registra el kernel con un nombre personalizable
python -m ipykernel install --user \\
  --name=aprendizajecomp \\
  --display-name="Python (AprendizajeComp)"


```

### Instalar dependencias
!pip install pandas numpy matplotlib scikit-learn joblib imbalanced-learn
Nota ejecutar solo una vez.

In [ ]:
!pip install pandas numpy matplotlib scikit-learn joblib imbalanced-learn

In [ ]:
import pandas as pd
import matplotlib as plt
from sklearn import tree
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
wine = pd.read_csv('winequality-red.csv')
wine.head(5)

In [ ]:
wine.quality.value_counts()


### identificacion de variables x, y 
como podemos ver en el dataset tenemos (1599,12), es decir 1599 filas y 12 columnas de las cuales 11 son datos que podemos considerar de entrada y el ultimo podemos considerar de salida.

ahora bien analizando el dataset podemos observar que tenemos varios niveles de calidad(observe la celda anterior), Ademas de que el dataset se encuentra desbalanceado.

esto lo que va a generar es que nuestro modelo no se entrene de la mejor manera, Cuando ejecutas wine.quality.value_counts(), ves que hay muchas más muestras de calidad 5-6 que de 3-4 o 8. El modelo tiende a aprender patrones de las clases mayoritarias (sobreajuste) e ignora las minorías. Resultado: predice mal los vinos buenos o malos. Por eso usamos SMOTE para generar sintéticamente más muestras de clases minoritarias, equilibrando el entrenamiento.

Ademas, "Simplificamos el problema de 6 clases a 2 clases binarias: vinos de baja calidad (≤5) vs alta calidad (>5). Esto reduce la complejidad del modelo y mejora el balance de clases."

In [ ]:
# lo primero que vamos hacer es a simplificar las clases 
wine['quality_binary'] = (wine['quality'] >= 6).astype(int)
print(wine['quality_binary'].value_counts())


### explicacion de simplificacion

```python
wine['quality_binary'] = (wine['quality'] >= 6).astype(int)
                         │────────────────────────────────│
                         Parte que crea 0s y 1s
```

Parte 1 : ``` wine['quality'] >= 6 ```

```python
    # Si quality es:
3  → 3 >= 6?  → FALSE
4  → 4 >= 6?  → FALSE
5  → 5 >= 6?  → FALSE
6  → 6 >= 6?  → TRUE  ✓
7  → 7 >= 6?  → TRUE  ✓
8  → 8 >= 6?  → TRUE  ✓
9  → 9 >= 6?  → TRUE  ✓
```

``Resultado`` una lista de true y false
```
Original:  [3, 5, 6, 7, 8, 3, 6, ...]
Resultado: [False, False, True, True, True, False, True, ...]
```

Parte 2 ``` .astype(int) ```

```
True  → 1
False → 0
```

al final queda un arreglo de 0 y 1 0 = calidad baja 1 = calidad alta


In [ ]:
wine.quality_binary.value_counts()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# hacemos uso de random_state eso lo que significa es que siempre escoge el mismo conjunto de datos para las pruebas.
train, test = train_test_split(wine, test_size=0.30, random_state=42)
train.shape

In [ ]:
test.shape


In [ ]:
x_train = train[['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']]
y_train = train['quality_binary']

In [ ]:
# Balanceo con SMOTE
from imblearn.over_sampling import SMOTE

smote = SMOTE(k_neighbors=2, random_state=42)
x_train_balanced, y_train_balanced = smote.fit_resample(x_train, y_train)

In [ ]:
y_train_balanced.value_counts()

In [ ]:
x_test = test[['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol']]
y_test = test['quality_binary']

### ¿Por qué Gradient Boosting Classifier?

**Gradient Boosting** es un algoritmo de ensamble que construye múltiples árboles de decisión secuencialmente, donde cada árbol aprende de los errores del anterior. 

**Características clave para nuestro problema:**

1. **Captura relaciones complejas**: mi dataset tiene 11 características con relaciones no lineales que un árbol simple no puede capturar
2. **Robusto ante desbalances**: Aunque este usando SMOTE para balancear, Gradient Boosting maneja bien clases desbalanceadas
3. **Excelente en clasificación binaria**: es ideal para este algoritmo ya que solo estamos manejando 0=calidad baja 1=calidad alta
4. **Auto-corrección**: Cada nuevo árbol corrige errores anteriores, mejorando iterativamente la precisión

**Parámetros usados:**
- `n_estimators=100`: 100 árboles que se corrigen mutuamente
- `learning_rate=0.1`: Aprendizaje moderado para evitar sobreajuste
- `max_depth=5`: Árboles pequeños para generalizar mejor
- `random_state=42`: Asegura reproducibilidad. Con este valor fijo, el modelo siempre genera los mismos resultados, permitiendo comparar experimentos y validar que los cambios en el rendimiento se deben al algoritmo, no a la aleatoridad.

In [ ]:
model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5,random_state=42)
model.fit(x_train_balanced, y_train_balanced)

In [ ]:
y_pred = model.predict(x_test)
y_pred

In [ ]:
df_prediccion = pd.DataFrame(y_pred)
df_prediccion['real']=y_test.values
df_prediccion.head(10)

In [ ]:
model.score(x_test, y_test)*100

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

# Predicciones
y_pred = model.predict(x_test)
y_proba = model.predict_proba(x_test)[:, 1]

# Metricas principales
accuracy = accuracy_score(y_test, y_pred)*100
precision = precision_score(y_test, y_pred)*100
recall = recall_score(y_test, y_pred)*100
f1 = f1_score(y_test, y_pred)*100   
roc_auc = roc_auc_score(y_test, y_proba)*100

print("=== Metricas de calidad del modelo ===")
print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1-Score : {f1:.3f}")
print(f"ROC-AUC  : {roc_auc:.3f}")

cm = confusion_matrix(y_test, y_pred)
print("\nMatriz de confusion:")
print(cm)

print("\nReporte de clasificacion:")
print(classification_report(y_test, y_pred, target_names=["Baja calidad (0)", "Alta calidad (1)"]))

ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Baja", "Alta"]).plot(cmap="Blues")
plt.title("Matriz de confusion - Gradient Boosting")
plt.show()

### Exportar un unico pipeline para 

Se crea el modelo (`modelo_vino_pipeline_v1.pkl`) Para usarlo desde app.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import joblib

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    ))
])

pipeline.fit(x_train_balanced, y_train_balanced)

joblib.dump(pipeline, "modelo_vino_pipeline_v1.pkl")

print("Listo: modelo_vino_pipeline_v1.pkl generado.")
print(f"Accuracy test: {pipeline.score(x_test, y_test) * 100:.2f}%")


### Análisis de resultados del modelo (Gradient Boosting)

La matriz de confusión muestra que el modelo clasifica correctamente la mayoría de los casos:

- Baja calidad correctamente detectada: 164
- Alta calidad correctamente detectada: 216
- Errores en baja calidad (falsos positivos): 49
- Errores en alta calidad (falsos negativos): 51

A partir de estos valores:

- **Accuracy ≈ 79.2%**: De los 480 vinos en total, el modelo acertó en 380 clasificaciones. Es el porcentaje general de aciertos, pero no dice cómo se distribuyen los errores entre clases.

- **Precision (clase Alta) ≈ 81.5%**: De los vinos que el modelo predijo como "Alta calidad", el 81.5% realmente eran de alta calidad. Significa que si el modelo dice que un vino es bueno, tienes 81.5% de confianza en que está en lo correcto.

- **Recall (clase Alta) ≈ 80.9%**: De todos los vinos que realmente son de "Alta calidad", el modelo detectó el 80.9%. Significa que de cada 100 vinos realmente buenos, encuentras 81 (pero pierdes 19 que confundes con bajos).

- **F1-score (clase Alta) ≈ 81.2%**: Es el balance perfecto entre Precision y Recall. Combina ambas métricas en un solo valor: si uno fuera muy alto y otro muy bajo, F1 sería bajo alertándote del problema. En tu caso, ambas están balanceadas en 81.2%.

### Interpretación

1. El modelo tiene un desempeño **bueno y equilibrado** para ambas clases.
2. La precisión alta indica que, cuando predice “Alta”, en la mayoría de casos acierta.
3. El recall alto indica que detecta bien los vinos realmente de calidad alta.
4. Los errores están bastante balanceados (49 vs 51), por lo que no hay sesgo fuerte hacia una sola clase.
5. En comparación con usar solo accuracy, estas métricas muestran de forma más completa la calidad real del modelo.